# Basic usage

We show how to use BILPEC for constrained ensemble pruning and feasibility.


We create a random set of samples.


In [ ]:
import numpy as np

n_models = 100
n_samples = 1000

rng = np.random.default_rng(42)

# Random Predictions
random_predictions = rng.random((n_models, n_samples))
# Random Labels
random_labels = rng.random(n_samples) > 0.5
# Low measures on first models, high on last
measures = np.arange(n_models)

We create a constraint on base learners.


In [ ]:
from bilpec.bilpec import MeasureConstraint

constraint = MeasureConstraint(
    measure_name="range",
    measures=measures,
    below_threshold=True,
    threshold_type="total",
    threshold=400,
)

constraints = [constraint]


We instantiate BILPEC.


In [ ]:
from bilpec.bilpec import BILPEC

bilpec = BILPEC(
    # The total number of models.
    ensemble_size=n_models,
    problem_type="binary",
    # The size of the pruned ensemble.
    fixed_ensemble_size=10,
    constraints=constraints,
    solver="CBC",
    solver_time_limit=1,
)


We fit BILPEC.


In [ ]:
bilpec.fit(predictions=random_predictions, labels=random_labels)

Here we see if the problem was feasible.


In [ ]:
bilpec.is_feasible()

Here we see the weights learned by BILPEC.


In [ ]:
bilpec.weights_

We can then use BILPEC to predict using soft voting.


In [ ]:
bilpec.predict_proba(predictions=random_predictions)

For other baselines.


In [ ]:
from autogluon.core.metrics import log_loss
from autogluon.core.models.greedy_ensemble.ensemble_selection import EnsembleSelection

from bilpec.baselines import SHERLOCK, HardVoting, SingleBest, SoftVoting

ges = EnsembleSelection(
    ensemble_size=n_models,
    problem_type="binary",
    metric=log_loss,
)

hard_voting = HardVoting(
    ensemble_size=n_models,
    problem_type="binary",
    metric=log_loss,
)

soft_voting = SoftVoting(
    ensemble_size=n_models,
    problem_type="binary",
    metric=log_loss,
)

single_best = SingleBest(
    ensemble_size=n_models,
    problem_type="binary",
    metric=log_loss,
)

sherlock: SHERLOCK = SHERLOCK(
    ensemble_size=n_models,
    problem_type="binary",
    constraints=constraints,
    max_steps=100,
)

baselines = [ges, soft_voting, single_best, hard_voting, sherlock]

for model in baselines:
    model.fit(predictions=random_predictions, labels=random_labels)
    model.predict_proba(random_predictions)